In [ ]:
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-zduwmfyx
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-zduwmfyx
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 173e636d9a3cb81c6101a160c1c8c86864eaaf5b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nvcc4jupyter: filename=nvcc4jupyter-1.2.2-py3-none-any.whl size=12545 sha256=f293982cdc354d66d92473c4350316b6ff6623d7028219d8cf93377ffb402266
  Stored in directory: /tmp/pip-ephem-wheel-cache-gm9f948s/wheels/2f/1e/79/584a8d85ad018ecc510baa90d4fbc161bd56a4d66943d5b0d4
Successfully built nvcc4jupyter
Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpcn4b939b".


In [6]:

%%writefile reduction_unroll8_16_simple.cu

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define CHECK(call) { \
    const cudaError_t error = call; \
    if (error != cudaSuccess) { \
        printf("Error: %s:%d, ", __FILE__, __LINE__); \
        printf("code:%d, reason: %s\n", error, cudaGetErrorString(error)); \
        exit(1); \
    } \
}

#define BDIM 512

// ----------- Unroll-8 kernel -----------
__global__ void reduceUnrolling8(int *g_idata, int *g_odata, unsigned int n)
{
    // Convert global data pointer to local block pointer
    int *idata = g_idata + blockIdx.x * blockDim.x * 8;

    unsigned int tid = threadIdx.x;
    unsigned int idx = blockIdx.x * blockDim.x * 8 + tid;

    // Unrolling 8
    int sum = 0;

    for (int i = 0; i < 8; i++) {
        unsigned int pos = idx + i * blockDim.x;

        if (pos < n)
            sum += g_idata[pos];
    }

    idata[tid] = sum;

    __syncthreads();

    // In-place reduction
    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride)
            idata[tid] += idata[tid + stride];

        __syncthreads();
    }

    // Write result of this block
    if (tid == 0)
        g_odata[blockIdx.x] = idata[0];
}

// ----------- Unroll-16 kernel -----------
__global__ void reduceUnrolling16(int *g_idata, int *g_odata, unsigned int n)
{
    // Convert global data pointer to local block pointer
    int *idata = g_idata + blockIdx.x * blockDim.x * 16;

    unsigned int tid = threadIdx.x;
    unsigned int idx = blockIdx.x * blockDim.x * 16 + tid;

    // Unrolling 16
    int sum = 0;

    for (int i = 0; i < 16; i++) {
        unsigned int pos = idx + i * blockDim.x;

        if (pos < n)
            sum += g_idata[pos];
    }

    idata[tid] = sum;

    __syncthreads();

    // In-place reduction
    for (int stride = blockDim.x / 2; stride > 0; stride >>= 1) {
        if (tid < stride) {
            idata[tid] += idata[tid + stride];
        }

        __syncthreads();
    }

    // Write result of this block
    if (tid == 0)
        g_odata[blockIdx.x] = idata[0];
}

// ----------- CPU reference reduction -----------
int cpuReduce(int *h_idata, int size)
{
    int sum = 0;

    for (int i = 0; i < size; i++)
        sum += h_idata[i];

    return sum;
}

// ----------- Main function -----------
int main()
{
    int size = 1 << 24;  // 16M elements
    int bytes = size * sizeof(int);

    // Allocate and initialize host array
    int *h_idata = (int *)malloc(bytes);

    if (h_idata == NULL) {
        printf("Host memory allocation failed!\n");
        return 1;
    }

    for (int i = 0; i < size; i++)
        h_idata[i] = rand() & 0xFF;

    int *d_idata, *d_odata;

    CHECK(cudaMalloc((void **)&d_idata, bytes));

    // Grid sizes for unroll-8 and unroll-16
    int grid8 = (size + BDIM * 8 - 1) / (BDIM * 8);
    int grid16 = (size + BDIM * 16 - 1) / (BDIM * 16);

    CHECK(cudaMalloc((void **)&d_odata, grid8 * sizeof(int)));

    // ---------------- CPU reduction ----------------
    cudaEvent_t start, stop;

    CHECK(cudaEventCreate(&start));
    CHECK(cudaEventCreate(&stop));

    CHECK(cudaEventRecord(start));

    int cpu_sum = cpuReduce(h_idata, size);

    CHECK(cudaEventRecord(stop));
    CHECK(cudaEventSynchronize(stop));

    float cpuTime;

    CHECK(cudaEventElapsedTime(&cpuTime, start, stop));

    // ---------------- GPU Unroll-8 ----------------
    CHECK(cudaMemcpy(d_idata, h_idata, bytes, cudaMemcpyHostToDevice));

    CHECK(cudaEventRecord(start));

    reduceUnrolling8<<<grid8, BDIM>>>(d_idata, d_odata, size);

    cudaError_t err8 = cudaGetLastError();

    if (err8 != cudaSuccess) {
        printf("Kernel launch error (Unroll-8): %s\n",
               cudaGetErrorString(err8));
        return -1;
    }

    CHECK(cudaDeviceSynchronize());

    CHECK(cudaEventRecord(stop));
    CHECK(cudaEventSynchronize(stop));

    float gpuTime8;

    CHECK(cudaEventElapsedTime(&gpuTime8, start, stop));

    int *h_odata = (int *)malloc(grid8 * sizeof(int));

    if (h_odata == NULL) {
        printf("Output memory allocation failed!\n");
        return 1;
    }

    CHECK(cudaMemcpy(h_odata, d_odata,
                     grid8 * sizeof(int), cudaMemcpyDeviceToHost));

    int gpu_sum8 = 0;

    for (int i = 0; i < grid8; i++)
        gpu_sum8 += h_odata[i];

    // ---------------- GPU Unroll-16 ----------------
    CHECK(cudaMemcpy(d_idata, h_idata, bytes, cudaMemcpyHostToDevice));

    CHECK(cudaEventRecord(start));

    reduceUnrolling16<<<grid16, BDIM>>>(d_idata, d_odata, size);

    cudaError_t err16 = cudaGetLastError();

    if (err16 != cudaSuccess) {
        printf("Kernel launch error (Unroll-16): %s\n",
               cudaGetErrorString(err16));
        return -1;
    }

    CHECK(cudaDeviceSynchronize());

    CHECK(cudaEventRecord(stop));
    CHECK(cudaEventSynchronize(stop));

    float gpuTime16;

    CHECK(cudaEventElapsedTime(&gpuTime16, start, stop));

    CHECK(cudaMemcpy(h_odata, d_odata,
                     grid16 * sizeof(int), cudaMemcpyDeviceToHost));

    int gpu_sum16 = 0;

    for (int i = 0; i < grid16; i++)
        gpu_sum16 += h_odata[i];

    // ---------------- Print results ----------------
    printf("\n========== REDUCTION RESULTS ==========\n");

    printf("CPU sum    : %d, time %.5f sec\n",
           cpu_sum, cpuTime / 1000.0f);

    printf("GPU sum-8  : %d, time %.5f ms\n",
           gpu_sum8, gpuTime8);

    printf("GPU sum-16 : %d, time %.5f ms\n",
           gpu_sum16, gpuTime16);

    printf("\n========== VERIFICATION ==========\n");

    printf("Unroll-8  : %s\n",
           (cpu_sum == gpu_sum8) ? "PASSED" : "FAILED");

    printf("Unroll-16 : %s\n",
           (cpu_sum == gpu_sum16) ? "PASSED" : "FAILED");

    // ---------------- Cleanup ----------------
    free(h_idata);
    free(h_odata);

    cudaFree(d_idata);
    cudaFree(d_odata);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}


Overwriting reduction_unroll8_16_simple.cu


In [7]:
!nvcc -arch=sm_75 reduction_unroll8_16_simple.cu -o reduction
!./reduction


========== REDUCTION RESULTS ==========
CPU sum    : 2139353471, time 0.04346 sec
GPU sum-8  : 2139353471, time 0.36765 ms
GPU sum-16 : 2139353471, time 0.29683 ms

========== VERIFICATION ==========
Unroll-8  : PASSED
Unroll-16 : PASSED
